In [7]:
import pandas as pd
import numpy as np
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA   

df = pd.read_csv('student-scores.csv')
df_clean = df.drop(columns=['id', 'first_name', 'last_name', 'email','gender','part_time_job','extracurricular_activities','career_aspiration'])

numeric_features = df_clean.select_dtypes(include=['int64', 'float64']).columns
categorical_features = df_clean.select_dtypes(include=['object', 'bool']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)])

clustering_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components=0.95)),  
    ('clusterer', AgglomerativeClustering(n_clusters=3))
])

df['cluster_label'] = clustering_pipeline.fit_predict(df_clean)
joblib.dump(clustering_pipeline, 'student_clustering_pipeline.pkl')
print(f"Cluster Distribution:\n{df['cluster_label'].value_counts()}")

Cluster Distribution:
cluster_label
0    1645
1     190
2     165
Name: count, dtype: int64


In [8]:
# 1. Run the prediction to get the labels
labels = clustering_pipeline.fit_predict(df_clean)

# 2. Add the labels to your original dataframe
df['cluster_label'] = labels

# 3. Save it as the specific filename the app is looking for
df.to_csv('student_scores_with_clusters.csv', index=False)

print("File saved successfully!")

File saved successfully!
